# Clinical Insulin ML Pipeline — Reprocessing + Training

This notebook runs the full end-to-end **clinical insulin regression** pipeline:

- Loads the SmartSensor CSV
- Reprocesses / feature engineers / splits (group-wise by patient)
- Trains multiple models and selects the best by **test RMSE**
- Saves:
  - **Best model bundle** for the backend (`outputs/best_model/inference_bundle.joblib`)
  - **Evaluation metrics** CSVs (`outputs/clinical_insulin_pipeline/latest/evaluation/`)
  - **Visualizations** (`outputs/clinical_insulin_pipeline/latest/models/`)

## What the app will use
The FastAPI backend loads:
- `outputs/best_model/inference_bundle.joblib`

So after this notebook finishes, the app can immediately serve `/api/recommend`.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


def _detect_backend_src(start: Path) -> Path:
    """Find backend/src either in this project or one level down."""
    p = start.resolve()
    while True:
        # Case A: notebook started inside Clinical-Insulin-Recommendation/
        cand = p / "backend" / "src"
        if cand.is_dir():
            return cand

        # Case B: notebook started from a parent workspace (e.g. E:\Glucosense app)
        cand2 = p / "Clinical-Insulin-Recommendation" / "backend" / "src"
        if cand2.is_dir():
            return cand2

        if p == p.parent:
            break
        p = p.parent

    raise RuntimeError(
        "Cannot locate backend/src. Start Jupyter in the project folder, "
        "or ensure 'Clinical-Insulin-Recommendation/backend/src' exists."
    )


SRC = _detect_backend_src(Path.cwd())
# repo root is the parent of backend/
repo_root = SRC.parent.parent

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Detected repo root:", repo_root)
print("SRC:", SRC)
print("Python:", sys.version.split()[0])


Detected repo root: D:\End_of_Year_Final_Project\Final-Year-Full-Project\Clinical-Insulin-Recommendation
SRC: D:\End_of_Year_Final_Project\Final-Year-Full-Project\Clinical-Insulin-Recommendation\backend\src
Python: 3.14.3


In [1]:
from clinical_insulin_pipeline.training import run_training
from clinical_insulin_pipeline.config import DEFAULT_DATA_CSV, repo_root_from_here

repo = repo_root_from_here()
print("Pipeline repo root:", repo)

csv_path = (repo / DEFAULT_DATA_CSV).resolve()
print("Training CSV:", csv_path)
print("CSV exists:", csv_path.is_file())

# Optional knobs
SKIP_LEARNING_CURVE = False
SKIP_SHAP = True  # SHAP can be heavy / optional on Windows


ModuleNotFoundError: No module named 'clinical_insulin_pipeline'

In [ ]:
# (removed duplicate cell)


In [3]:
# Run training (preprocess → train models → evaluate → export bundle + metrics + plots)
res = run_training(
    csv_path,
    skip_learning_curve=SKIP_LEARNING_CURVE,
    skip_shap=SKIP_SHAP,
)

res.leaderboard.head(10), res.best_name, res.test_metrics, str(res.output_dir)


INFO random_forest test RMSE=3.4271 MAE=2.9882 R2=-0.0201
INFO hist_gradient_boosting test RMSE=3.4177 MAE=2.9955 R2=-0.0146
INFO xgboost test RMSE=3.5816 MAE=3.0498 R2=-0.1142
INFO Saved evaluation CSVs under E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\clinical_insulin_pipeline\latest\evaluation
INFO Deployed bundle to E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\best_model\inference_bundle.joblib


(                    model       mae      rmse          mape        r2  \
 0  hist_gradient_boosting  2.995466  3.417715  7.347365e+07 -0.014555   
 1           random_forest  2.988239  3.427054  7.342717e+07 -0.020108   
 2                 xgboost  3.049848  3.581590  7.443707e+07 -0.114181   
 
    max_error  
 0   5.980205  
 1   6.212742  
 2   8.039640  ,
 'hist_gradient_boosting',
 {'mae': 2.995465930226316,
  'rmse': 3.4177145401316573,
  'mape': 73473650.93316962,
  'r2': -0.014555452148053982,
  'max_error': 5.980205492800214},
 'E:\\Glucosense app\\Clinical-Insulin-Recommendation\\outputs\\clinical_insulin_pipeline\\latest')

In [4]:
# Where artifacts were saved
from insulin_system.persistence.bundle import resolve_inference_bundle_path

latest_dir = res.output_dir
print("Run output dir:", latest_dir)
print("Leaderboard:", latest_dir / "leaderboard.csv")
print("Eval metrics:", latest_dir / "evaluation" / "model_metrics.csv")
print("Eval summary:", latest_dir / "evaluation" / "evaluation_summary.csv")
print("Plots dir:", latest_dir / "models")

best_bundle_path = resolve_inference_bundle_path(None)
print("Backend bundle path (should exist):", best_bundle_path)
print("Bundle exists:", best_bundle_path.is_file())


Run output dir: E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\clinical_insulin_pipeline\latest
Leaderboard: E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\clinical_insulin_pipeline\latest\leaderboard.csv
Eval metrics: E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\clinical_insulin_pipeline\latest\evaluation\model_metrics.csv
Eval summary: E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\clinical_insulin_pipeline\latest\evaluation\evaluation_summary.csv
Plots dir: E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\clinical_insulin_pipeline\latest\models
Backend bundle path (should exist): E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\best_model\inference_bundle.joblib
Bundle exists: True


In [5]:
# Verify the backend can load the saved bundle
from insulin_system.persistence import load_best_model

bundle = load_best_model()
print("Loaded bundle:", bundle.path)
print("Model name:", bundle.model_name)
print("n_features:", len(bundle.feature_names))
print("Test metrics in bundle:", bundle.data.get("test_metrics"))


INFO Loaded clinical insulin bundle from E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\best_model\inference_bundle.joblib


Loaded bundle: E:\Glucosense app\Clinical-Insulin-Recommendation\outputs\best_model\inference_bundle.joblib
Model name: hist_gradient_boosting
n_features: 22
Test metrics in bundle: {'mae': 2.995465930226316, 'rmse': 3.4177145401316573, 'mape': 73473650.93316962, 'r2': -0.014555452148053982, 'max_error': 5.980205492800214}


## Notes

- If `CSV exists` is `False`, place your dataset at `Clinical-Insulin-Recommendation/data/SmartSensor_DiabetesMonitoring.csv` (or change `csv_path`).
- If you want a faster run, set:
  - `SKIP_LEARNING_CURVE = True`
  - `SKIP_SHAP = True`

After training, start the backend and the UI:

```bash
# from Clinical-Insulin-Recommendation/
python -m uvicorn backend.app:app --reload --port 8000

# in another terminal
cd frontend
npm run dev
```
